# 8.3 - Embeddings & Vector Stores

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook works through Embeddings & Vector Stores hands-on: Embedding APIs: text to vectors; ChromaDB: a local vector store for prototyping; AlloyDB pgvector: SQL and vectors in one database; Distance metrics: prove cosine equals dot product on normalized vectors.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install dependencies

One commented `pip install` line for everything the runnable cells touch: the Gemini client, numpy for the vector math, Chroma for the local store. Uncomment it on Colab or a fresh machine; skip it if these are already installed.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai numpy chromadb python-dotenv -q  # noqa


**Explanation**

A pure environment-prep cell - it installs libraries and does nothing else. It stays commented so re-running the notebook locally never re-triggers a slow install you don't need.

**How the code works - step by step**
- **`google-genai`** - the current Gemini SDK, used for `embed_content` in cell 3.
- **`numpy`** - array math for the cosine-similarity matrix and normalization checks.
- **`chromadb`** - the embedded vector database used in cell 4.
- **`python-dotenv`** - lets you load keys from a `.env` file instead of hardcoding them.
- **`# noqa`** - silences the linter on the `!pip` shell magic.

**In one line:** uncomment once on a fresh env, then leave it be.

**What you'll see:** (no output - environment prep)

## Setup - load your API key

The embedding calls need a Google AI Studio key. This cell wires it in from the environment without ever writing the secret into the notebook.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`, from aistudio.google.com). Any one provider key is enough to start.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A key-loading cell, not a model call. `setdefault` only sets `GOOGLE_API_KEY` to an empty string if it isn't already present, so a real key you exported in your shell (or loaded via dotenv) is left untouched - the empty default just keeps later code from crashing on a missing variable.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - registers the variable name, but does NOT overwrite a key already set in your environment.
- The comment points you at where to get the key and reminds you never to hardcode it.

**In one line:** make sure the key variable exists, without pasting a secret into the file.

**What you'll see:** (no output - environment prep)

## 1 - Embedding APIs: text to vectors

Every downstream store is only as good as the vectors you feed it. This cell locks in the three habits that survive every later migration: batch the call, declare the `task_type`, and pick dimensions deliberately via Matryoshka truncation. It embeds four short phrases and prints their pairwise similarities so you can watch semantics fall out of the numbers.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# EMBEDDING APIs - Gemini vs open-source
# pip install google-genai numpy
from google import genai
from google.genai import types
import numpy as np, time

client = genai.Client()  # reads GOOGLE_API_KEY from the environment
EMBED = "gemini-embedding-001"

def embed_batch(texts, task="RETRIEVAL_DOCUMENT", dims=768):
    """ONE call for the whole list. task_type matters; dims uses Matryoshka truncation."""
    r = client.models.embed_content(
        model=EMBED, contents=texts,
        config=types.EmbedContentConfig(task_type=task, output_dimensionality=dims))
    return [np.array(e.values) for e in r.embeddings]

# Open-source alternative (local, free, private):
# from sentence_transformers import SentenceTransformer
# local = SentenceTransformer("BAAI/bge-m3"); local.encode(texts)

texts = ["machine learning algorithms", "AI and deep learning models",
         "cooking biryani recipe", "refund and return policy"]

start = time.time()
embs = embed_batch(texts)
elapsed = (time.time() - start) * 1000
print(f"{EMBED}: {len(embs)} texts, {len(embs[0])} dims, {elapsed:.0f}ms (one batched call)\n")

print("Similarity matrix:")
for i, t1 in enumerate(texts):
    row = f"  {t1[:24]:24s}"
    for j in range(len(texts)):
        sim = np.dot(embs[i], embs[j]) / (np.linalg.norm(embs[i]) * np.linalg.norm(embs[j]))
        row += f" {sim:.2f}"
    print(row)

# Output:
#   gemini-embedding-001: 4 texts, 768 dims, 240ms (one batched call)
#   Similarity matrix:
#   machine learning algorith  1.00 0.78 0.31 0.35
#   AI and deep learning mode  0.78 1.00 0.30 0.33
#   cooking biryani recipe     0.31 0.30 1.00 0.28
#   refund and return policy   0.35 0.33 0.28 1.00
#   ML and AI: high similarity. ML and biryani: low. That is semantic search.

# The 2026 menu (prices are per MILLION tokens - check current rates):
#   gemini-embedding-001   3072 dims (Matryoshka)  about $0.15/M   API, MTEB leader
#   OpenAI 3-small         1536 dims               about $0.02/M   API, cheap default
#   OpenAI 3-large         3072 dims               about $0.13/M   API
#   Qwen3-Embedding-8B     4096 dims               Free       Local, Apache 2.0
#   BGE-M3                  1024 dims               Free       Local, 100+ languages

**Explanation**

This is a measurement harness wrapped around one API call. `embed_batch` sends the whole list of texts in a single `embed_content` request, tags them with a task type, and truncates to 768 dimensions; the rest of the cell computes a cosine-similarity matrix by hand so you can see that meaning-related phrases score high and unrelated ones score low.

**How the code works - step by step**
- **`client = genai.Client()`** - reads `GOOGLE_API_KEY` from the environment; no key passed in code.
- **`embed_batch(texts, task, dims)`** - ONE `embed_content` call for the entire list; `task_type="RETRIEVAL_DOCUMENT"` stamps these as documents (queries would use `RETRIEVAL_QUERY`), and `output_dimensionality=768` is the Matryoshka truncation knob.
- **commented `SentenceTransformer` block** - the local, free, private open-source alternative (BGE-M3) if data can't leave your machine.
- **timing + `np.dot / (norm*norm)` loop** - times the single batched call, then builds the 4x4 cosine matrix.
- **trailing comment table** - a 2026 price/dimension menu for Gemini, OpenAI, Qwen3, and BGE-M3.

**In one line:** batch once, tag the task, truncate the dims, then read meaning off the similarity matrix.

**What you'll see:** Prints the model name, count, dims and latency for one batched call (e.g. `4 texts, 768 dims, 240ms`), then a similarity matrix where the two ML/AI phrases score ~0.78 to each other but ~0.30 against "biryani" and "refund" - that gap is semantic search working.

## 2 - ChromaDB: a local vector store for prototyping

A plain Python list forgets everything on restart and scans every vector per query. Chroma is the smallest step up that fixes both: an embedded database (it lives inside your Python process, no server) with an ANN index, metadata filtering, and optional persistence - and it's a two-line install.

In [ ]:
# CHROMADB - Local vector database
# pip install chromadb
import chromadb

client = chromadb.Client()  # in-memory; chromadb.PersistentClient(path="./db") to keep it
collection = client.create_collection("netsetos_docs")

# Chroma auto-embeds with its DEFAULT local model (all-MiniLM) - fine for a demo.
# In production pass your own embeddings so index and query use the SAME model.
docs = [
    "Netsetos offers GenAI courses in Hyderabad. 14 modules, 146 hours.",
    "Refund policy: full refund within 7 days. 50% from 8-30 days.",
    "GenAI course costs 14999 rupees. Includes Discord, live sessions, certificate.",
    "Live classes daily at 7 PM IST on YouTube. 85% completion rate.",
    "Prerequisites: basic Python and high school math. No ML experience needed.",
]

collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))],
    metadatas=[{"source": f"faq_{i}.txt"} for i in range(len(docs))],
)

print(f"Indexed: {collection.count()} documents\n")

# Search by natural language
for q in ["How much does the course cost?", "Can I get a refund?", "Do I need ML experience?"]:
    results = collection.query(query_texts=[q], n_results=2)
    print(f"  Q: {q}")
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        print(f"    [{dist:.3f}] {doc[:70]}...")
    print()

# Metadata filtering: narrow the candidates BEFORE the vector math
hits = collection.query(query_texts=["course price"], n_results=2,
                        where={"source": "faq_2.txt"})
print(f"filtered: {hits['documents'][0][0][:60]}...")

print("ChromaDB: 5 lines to index, 3 to search. Perfect for prototyping.")

# Output:
#   Indexed: 5 documents
#   (distances: LOWER = closer - the opposite direction of cosine similarity)
#   Q: How much does the course cost?
#     [0.512] GenAI course costs 14999 rupees. Includes Discord, live se...
#     [0.891] Netsetos offers GenAI courses in Hyderabad. 14 modules, 14...
#   filtered: GenAI course costs 14999 rupees. Includes Discord...

**Explanation**

This cell stands up a real vector store, indexes five FAQ documents, and runs natural-language queries against them - the whole index-then-search loop in about eight lines. Chroma auto-embeds with its bundled local model here (fine for a demo); the comment flags that in production you pass your own embeddings so index and query share one model.

**How the code works - step by step**
- **`chromadb.Client()`** - an in-memory store; the comment shows `PersistentClient(path=...)` to keep it on disk across restarts.
- **`create_collection` + `collection.add(...)`** - registers five docs with `ids` and per-doc `metadatas` (a `source` filename each).
- **`collection.count()`** - confirms what got indexed.
- **`collection.query(query_texts=[q], n_results=2)`** - searches by meaning for three questions; note distances are LOWER = closer, the opposite direction of cosine similarity.
- **`where={"source": "faq_2.txt"}`** - metadata pre-filtering: narrow the candidate set BEFORE the vector math.

**In one line:** add docs with metadata, query in natural language, and filter before searching.

**What you'll see:** Prints `Indexed: 5 documents`, then for each question the top-2 matches with their distances (the pricing doc surfaces first for "How much does the course cost?" at ~0.512), the metadata-filtered single hit, and a closing line about how few lines this took.

## 3 - AlloyDB pgvector: SQL and vectors in one database

When your app already has a relational database, vectors usually belong in it. pgvector adds a vector column type and similarity operators to Postgres (`<=>` is cosine distance: lower = closer), so one query can join "nearest chunks" with "WHERE tenant_id = 42 AND has_permission" - no second system to sync.

> **Needs an AlloyDB / Postgres instance to actually run** - shown here as read-along SQL.

In [ ]:
# ALLOYDB PGVECTOR - SQL + vector search
# pip install asyncpg pgvector sqlalchemy
# Requires AlloyDB instance on GCP

# Schema: documents table with vector column
CREATE_SQL = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS documents (
    id SERIAL PRIMARY KEY,
    content TEXT NOT NULL,
    source VARCHAR(255),
    embedding vector(768),  -- gemini-embedding-001 truncated to 768 (Matryoshka)
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
-- AlloyDB alternative: Google's ScaNN index - faster builds and a
-- smaller memory footprint than HNSW at large row counts.
"""

# Insert with embedding
INSERT_SQL = """
INSERT INTO documents (content, source, embedding) 
VALUES ($1, $2, $3::vector)
"""

# Search by similarity
SEARCH_SQL = """
SELECT content, source, 
       1 - (embedding <=> $1::vector) AS similarity
FROM documents
ORDER BY embedding <=> $1::vector
LIMIT $2
"""

print("AlloyDB pgvector Setup:\n")
print("  1. CREATE EXTENSION vector;")
print("  2. Add vector(768) column to your existing table")
print("  3. CREATE INDEX using hnsw (or ScaNN on AlloyDB)")
print("  4. Search: ORDER BY embedding <=> query_vector\n")
print("  Advantages:")
print("    - Same database for structured data AND vectors")
print("    - SQL joins: combine vector search with filters")
print("    - AlloyDB: GCP-managed, auto-scaling, 100x faster than vanilla PG")
print("    - Example: WHERE category='genai' ORDER BY embedding <=> query")

# Output:
#   AlloyDB pgvector Setup:
#   1. CREATE EXTENSION vector;
#   2. Add vector(768) column to your existing table
#   3. CREATE INDEX using hnsw (or ScaNN on AlloyDB)
#   4. Search: ORDER BY embedding <=> query_vector

**Explanation**

This is a reference cell, not a live database call - it defines the SQL you'd run and then prints a setup checklist. The three SQL strings are the whole pattern: create the extension and an HNSW-indexed vector column, insert rows with an embedding, and order by the cosine operator to search.

**How the code works - step by step**
- **`CREATE_SQL`** - `CREATE EXTENSION vector`, a `documents` table with an `embedding vector(768)` column (768 = Gemini truncated via Matryoshka), and an HNSW index (`m=16, ef_construction=64`); a comment notes AlloyDB's ScaNN index as the large-scale alternative.
- **`INSERT_SQL`** - parameterized insert casting the array to `::vector`.
- **`SEARCH_SQL`** - `ORDER BY embedding <=> $1::vector` for nearest neighbors, returning `1 - (embedding <=> ...)` as a similarity score.
- **`print(...)` block** - a four-step setup recap plus the advantages (one database for structured + vector data, SQL joins, GCP-managed autoscaling).

**In one line:** add a vector column and an HNSW index to a table you already have, then search with `ORDER BY embedding <=> query`.

**What you'll see:** No database is contacted - it prints the "AlloyDB pgvector Setup" checklist (create extension, add the vector column, create the HNSW/ScaNN index, search by the `<=>` operator) and the advantages list.

## 4 - Distance metrics: prove cosine equals dot product on normalized vectors

Which distance metric is a small decision that fails silently when wrong - retrieval still returns results, just subtly worse-ranked ones. The rule is mechanical: match the model card, and on L2-normalized vectors cosine similarity and dot product are the same number. This cell proves it in six lines.

In [ ]:
# Prove it in six lines: on NORMALIZED vectors, cosine == dot product
import numpy as np

a, b = np.random.rand(768), np.random.rand(768)
an, bn = a / np.linalg.norm(a), b / np.linalg.norm(b)   # normalize once at index time

cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"cosine(a,b)      = {cosine:.6f}")
print(f"dot(a_norm,b_norm)= {np.dot(an, bn):.6f}")  # identical - and cheaper per query

# Output:
#   cosine(a,b)      = 0.751592
#   dot(a_norm,b_norm)= 0.751592

**Explanation**

A tiny verification harness, not a model call. It makes two random 768-dim vectors, normalizes copies of them, then shows that the full cosine formula on the raw vectors and a bare dot product on the normalized ones print the identical value - which is why many stores just normalize once at index time and use the cheaper dot product per query.

**How the code works - step by step**
- **`a, b = np.random.rand(768), ...`** - two arbitrary vectors stand in for embeddings.
- **`an, bn = a/norm(a), b/norm(b)`** - normalize once (what you'd do at index time).
- **`cosine = np.dot(a,b)/(norm(a)*norm(b))`** - the full cosine-similarity formula on the raw vectors.
- **`np.dot(an, bn)`** - a plain dot product on the normalized copies - no division needed, so it's cheaper.

**In one line:** normalize once, and dot product gives you cosine for free.

**What you'll see:** Prints two numbers that match to six decimals (e.g. `cosine(a,b) = 0.751592` and `dot(a_norm,b_norm) = 0.751592`) - concrete proof the two metrics agree on normalized vectors.

Read end to end, these cells trace one retrieval stack from vector to store to search: cell 3 turns text into batched, task-typed, Matryoshka-truncatable embeddings; cell 4 parks them in Chroma with an index and metadata filter you can actually query; cell 5 shows the same idea as SQL columns and operators in Postgres/AlloyDB; and cell 6 proves the metric rule (normalize once, cosine and dot agree). Together they answer the sizing question the lesson opens with - which embedding, which home, which distance - by having you run the cheap end (Chroma, local math) and read along on the expensive end (pgvector, Vertex). Next up in 8.4 Advanced Retrieval, these tuned indexes become the substrate for hybrid search, reranking, and full RRF fusion.